In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
import os

project_path = "/content/drive/MyDrive/smart-waste-classifier"

os.makedirs(project_path, exist_ok=True)

print("Project folder created!")
print("Location:", project_path)

In [ ]:
import os

zip_path = "/content/garbage-classification.zip"

if os.path.exists(zip_path):
    print("Dataset ZIP found!")
    print("Size:", round(os.path.getsize(zip_path) / (1024 * 1024), 2), "MB")
else:
    print("Dataset ZIP not found.")

In [ ]:
import torch

print(torch.cuda.is_available())

In [ ]:
!pip install -q kaggle

In [ ]:
from getpass import getpass
import os

kaggle_token = getpass("Paste your Kaggle token: ")
os.environ["KAGGLE_API_TOKEN"] = kaggle_token

print("Kaggle token configured!")

In [ ]:
!kaggle datasets list -s "garbage classification"

In [ ]:
!kaggle datasets download -d mostafaabla/garbage-classification

In [ ]:
import zipfile

with zipfile.ZipFile("/content/garbage-classification.zip", "r") as zip_ref:
    zip_ref.extractall("/content/garbage_dataset")

print("Dataset extracted successfully!")

In [ ]:
import os

dataset_path = "/content/garbage_dataset"

for root, dirs, files in os.walk(dataset_path):
    level = root.replace(dataset_path, "").count(os.sep)

    if level > 2:
        continue

    print("  " * level + os.path.basename(root) + "/")

In [ ]:
import os

data_path = "/content/garbage_dataset/garbage_classification"

for category in sorted(os.listdir(data_path)):
    category_path = os.path.join(data_path, category)

    if os.path.isdir(category_path):
        count = len([
            f for f in os.listdir(category_path)
            if f.lower().endswith((".jpg", ".jpeg", ".png"))
        ])

        print(f"{category:15} : {count} images")

In [ ]:
import random
import matplotlib.pyplot as plt
import os
from PIL import Image

categories = sorted(os.listdir(data_path))

fig, axes = plt.subplots(12, 3, figsize=(12, 36))

for row, category in enumerate(categories):

    category_path = os.path.join(data_path, category)

    images = [
        f for f in os.listdir(category_path)
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ]

    random_images = random.sample(images, 3)

    for col, image_name in enumerate(random_images):

        image_path = os.path.join(category_path, image_name)

        img = Image.open(image_path).convert("RGB")

        axes[row, col].imshow(img)
        axes[row, col].set_title(category)
        axes[row, col].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import os
from PIL import Image
from collections import Counter

data_path = "/content/garbage_dataset/garbage_classification"

image_sizes = []

for category in os.listdir(data_path):

    category_path = os.path.join(data_path, category)

    if os.path.isdir(category_path):

        for filename in os.listdir(category_path):

            if filename.lower().endswith((".jpg", ".jpeg", ".png")):

                image_path = os.path.join(category_path, filename)

                with Image.open(image_path) as img:
                    image_sizes.append(img.size)

size_counts = Counter(image_sizes)

print("Total images checked:", len(image_sizes))
print("Number of unique image sizes:", len(size_counts))

print("\nMost common image sizes:")

for size, count in size_counts.most_common(10):
    print(f"{size}: {count} images")

if len(size_counts) == 1:
    print("\nAll images have the same dimensions!")
else:
    print("\nImages have different dimensions.")

In [ ]:
import os
import random
import matplotlib.pyplot as plt
from PIL import Image

trash_path = os.path.join(data_path, "trash")

trash_images = [
    file for file in os.listdir(trash_path)
    if file.lower().endswith((".jpg", ".jpeg", ".png"))
]

random.seed(42)
selected_images = random.sample(trash_images, 30)

fig, axes = plt.subplots(5, 6, figsize=(15, 13))

for i, image_name in enumerate(selected_images):
    image_path = os.path.join(trash_path, image_name)
    img = Image.open(image_path).convert("RGB")

    row = i // 6
    col = i % 6

    axes[row, col].imshow(img)
    axes[row, col].set_title(image_name[:18], fontsize=8)
    axes[row, col].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
class_mapping = {
    "battery": "battery",
    "biological": "biological",
    "brown-glass": "glass",
    "green-glass": "glass",
    "white-glass": "glass",
    "cardboard": "cardboard",
    "clothes": "clothes",
    "metal": "metal",
    "paper": "paper",
    "plastic": "plastic",
    "shoes": "shoes",
    "trash": "hygiene_waste"
}

print("Class mapping created!")
print("Number of original categories:", len(class_mapping))
print("Final categories:", sorted(set(class_mapping.values())))

In [ ]:
from collections import Counter
import os

class_counts = Counter()

for original_class, final_class in class_mapping.items():

    folder_path = os.path.join(data_path, original_class)

    image_files = [
        file for file in os.listdir(folder_path)
        if file.lower().endswith((".jpg", ".jpeg", ".png"))
    ]

    class_counts[final_class] += len(image_files)


print("Final class distribution:\n")

for category, count in sorted(class_counts.items()):
    print(f"{category:15s}: {count} images")

print("\nTotal images:", sum(class_counts.values()))
print("Total final classes:", len(class_counts))

In [ ]:
import os
from sklearn.model_selection import train_test_split
from collections import Counter

file_paths = []
labels = []

# Collect image paths and their final class labels
for original_class, final_class in class_mapping.items():

    folder_path = os.path.join(data_path, original_class)

    for filename in os.listdir(folder_path):

        if filename.lower().endswith((".jpg", ".jpeg", ".png")):

            image_path = os.path.join(folder_path, filename)

            file_paths.append(image_path)
            labels.append(final_class)


# First split: 70% training, 30% temporary
train_paths, temp_paths, train_labels, temp_labels = train_test_split(
    file_paths,
    labels,
    test_size=0.30,
    random_state=42,
    stratify=labels
)


# Second split: divide the temporary 30% equally
val_paths, test_paths, val_labels, test_labels = train_test_split(
    temp_paths,
    temp_labels,
    test_size=0.50,
    random_state=42,
    stratify=temp_labels
)


# Display split sizes
print("Dataset split completed!\n")

print("Training images:", len(train_paths))
print("Validation images:", len(val_paths))
print("Testing images:", len(test_paths))

print("\nTotal images:", len(train_paths) + len(val_paths) + len(test_paths))


# Display class distribution in each split
print("\nTraining class distribution:")
print(Counter(train_labels))

print("\nValidation class distribution:")
print(Counter(val_labels))

print("\nTesting class distribution:")
print(Counter(test_labels))

In [ ]:
import os
from collections import Counter
from PIL import Image

filename_counts = Counter()
image_size_counts = Counter()

for original_class in class_mapping:

    folder_path = os.path.join(data_path, original_class)

    for filename in os.listdir(folder_path):

        if filename.lower().endswith((".jpg", ".jpeg", ".png")):

            filename_counts[filename] += 1

            image_path = os.path.join(folder_path, filename)

            with Image.open(image_path) as img:
                image_size_counts[img.size] += 1


# Check repeated filenames
repeated_filenames = {
    name: count
    for name, count in filename_counts.items()
    if count > 1
}

print("Total images checked:", sum(filename_counts.values()))

print("\nRepeated filenames:", len(repeated_filenames))
print("Examples:", list(repeated_filenames.items())[:10])

print("\nNumber of unique image dimensions:", len(image_size_counts))
print("Most common image dimensions:")

for size, count in image_size_counts.most_common(10):
    print(size, ":", count, "images")

In [ ]:
import os
import hashlib
from collections import defaultdict

# Store all image information here
image_records = []

# Match every image path with its label and split
splits = {
    "train": (train_paths, train_labels),
    "validation": (val_paths, val_labels),
    "test": (test_paths, test_labels)
}


# Function to calculate the MD5 hash of an image file
def calculate_md5(file_path):

    md5 = hashlib.md5()

    with open(file_path, "rb") as file:
        for chunk in iter(lambda: file.read(4096), b""):
            md5.update(chunk)

    return md5.hexdigest()


# Calculate hashes for all images in all three splits
hash_groups = defaultdict(list)

for split_name, (paths, labels) in splits.items():

    for image_path, label in zip(paths, labels):

        image_hash = calculate_md5(image_path)

        hash_groups[image_hash].append({
            "path": image_path,
            "class": label,
            "split": split_name
        })


# Keep only hashes that belong to more than one file
duplicate_groups = {
    image_hash: records
    for image_hash, records in hash_groups.items()
    if len(records) > 1
}


# Count duplicate groups and files involved
duplicate_files = sum(
    len(records) for records in duplicate_groups.values()
)

cross_split_groups = {
    image_hash: records
    for image_hash, records in duplicate_groups.items()
    if len(set(record["split"] for record in records)) > 1
}


# Display summary
print("Total images checked:", sum(len(paths) for paths, labels in splits.values()))

print("\nDuplicate groups found:", len(duplicate_groups))
print("Total files involved in duplicates:", duplicate_files)

print("\nDuplicate groups across different splits:", len(cross_split_groups))


# Display details of every duplicate group
for group_number, (image_hash, records) in enumerate(
    duplicate_groups.items(), start=1
):

    classes = sorted(set(record["class"] for record in records))
    split_names = sorted(set(record["split"] for record in records))

    print(f"\nDuplicate Group {group_number}")
    print("MD5 Hash:", image_hash)
    print("Classes:", classes)
    print("Splits:", split_names)

    if len(split_names) > 1:
        print("WARNING: Duplicate images appear across different splits!")

    for record in records:
        print(
            f"  Class: {record['class']} | "
            f"Split: {record['split']} | "
            f"File: {os.path.basename(record['path'])}"
        )

In [ ]:
from sklearn.model_selection import train_test_split

# Store images that we decide to keep
clean_records = []

# Count files removed under each policy
removed_within_split = 0
removed_across_splits = 0
removed_conflicting_labels = 0


for image_hash, records in hash_groups.items():

    # Find the unique class labels in this duplicate group
    classes = set(record["class"] for record in records)

    # CASE 1: Same image has different class labels
    if len(classes) > 1:

        # Exclude all copies until we manually review their labels
        removed_conflicting_labels += len(records)

    # CASE 2: Same image has the same class label
    elif len(records) > 1:

        split_names = set(record["split"] for record in records)

        # Sort paths so the copy we keep is selected consistently
        sorted_records = sorted(records, key=lambda record: record["path"])

        # Keep one copy
        clean_records.append(sorted_records[0])

        # Count the extra copies that we remove
        removed_count = len(sorted_records) - 1

        if len(split_names) > 1:
            removed_across_splits += removed_count
        else:
            removed_within_split += removed_count

    # CASE 3: Image has no duplicates
    else:
        clean_records.append(records[0])


# Create cleaned path and label lists
clean_file_paths = [record["path"] for record in clean_records]
clean_labels = [record["class"] for record in clean_records]


# Display cleaning summary
total_removed = (
    removed_within_split
    + removed_across_splits
    + removed_conflicting_labels
)

print("Original images:", sum(len(paths) for paths, labels in splits.values()))

print("\nFiles removed within the same split:", removed_within_split)
print("Files removed across different splits:", removed_across_splits)
print("Files excluded due to conflicting labels:", removed_conflicting_labels)

print("\nTotal files removed:", total_removed)
print("Remaining images:", len(clean_file_paths))

In [ ]:
import os
import shutil

drive_data_path = os.path.join(project_path, "data", "raw")

os.makedirs(drive_data_path, exist_ok=True)

drive_zip_path = os.path.join(
    drive_data_path,
    "garbage-classification.zip"
)

shutil.copy2(zip_path, drive_zip_path)

print("Dataset backup saved to Google Drive!")
print("Location:", drive_zip_path)
print("Size:", round(os.path.getsize(drive_zip_path) / (1024 * 1024), 2), "MB")

In [ ]:
import os

if os.path.exists(drive_zip_path):
    print("Drive backup verified!")
    print("File size:", round(
        os.path.getsize(drive_zip_path) / (1024 * 1024), 2
    ), "MB")
else:
    print("Backup verification failed!")

In [ ]:
import os
import zipfile

# Location of the permanent ZIP backup in Google Drive
drive_zip_path = "/content/drive/MyDrive/smart-waste-classifier/data/raw/garbage-classification.zip"

# Temporary extraction location in Colab
extract_path = "/content/garbage_dataset"

# Extract only if the dataset isn't already available
data_path = os.path.join(extract_path, "garbage_classification")

if not os.path.exists(data_path):
    print("Extracting dataset from Google Drive...")

    with zipfile.ZipFile(drive_zip_path, "r") as zip_ref:
        zip_ref.extractall(extract_path)

    print("Dataset extracted successfully!")

else:
    print("Dataset already exists in this runtime!")

print("\nDataset location:", data_path)
print("Dataset exists:", os.path.exists(data_path))
print("Number of original classes:", len(os.listdir(data_path)))